# Machine learning con el TMT digital: cuaderno de práctica

En este cuaderno vamos a repasar los temas del curso usando **datos reales y públicos** de 74 personas que hicieron el Trail Making Test en una computadora (Juantorena et al., 2026, *Scientific Reports*): 41 con deterioro cognitivo leve (DCL) y 33 controles.

**No hace falta saber programar.**

- Cada bloque gris es una *celda de código*. Para ejecutarla, hacé clic adentro y apretá **Shift + Enter**.
- Ejecutá las celdas **en orden**, de arriba hacia abajo.
- Las celdas marcadas con ✏️ tienen un valor que podés cambiar para experimentar. Después de cambiarlo, volvé a ejecutar esa celda.

**Contenido**

1. Cargar los datos
2. Mirar los datos antes de modelar
3. Aprendizaje no supervisado
4. Un primer clasificador y el sobreajuste
5. Matriz de confusión y métricas
6. El umbral de decisión
7. Curva ROC y AUC
8. Validación cruzada y baselines
9. ¿Es mejor que el azar?
10. Regresión: predecir el MMSE

## 0. Preparación
Esta celda carga las herramientas que vamos a usar. Todas vienen instaladas en Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_predict, permutation_test_score
from sklearn import metrics

plt.rcParams.update({"figure.figsize": (6, 4), "font.size": 12})
COLORES = {"Control": "#42B9B2", "DCL": "#DE237B"}
print("Listo")

## 1. Cargar los datos

Los datos del estudio son públicos y están en OSF: https://osf.io/nm9xy/

Los archivos originales tienen nombres en inglés y más de 100 columnas. Esta celda los descarga, los une y se queda con las variables que vamos a usar, con nombres en castellano.

In [ ]:
OSF = "https://osf.io/download/"
digital = pd.read_csv(OSF + "38abz/")       # variables del TMT computarizado
demograficas = pd.read_csv(OSF + "6c9gu/")  # edad, sexo, educación y grupo
neuro = pd.read_csv(OSF + "3s2xz/")         # batería neuropsicológica

datos = (demograficas
         .merge(neuro.drop(columns="group"), on="subject_id")
         .merge(digital.drop(columns="group"), on="subject_id"))

NOMBRES = {
    "subject_id": "participante", "age": "edad", "years_of_education": "anios_educacion",
    "mmse": "mmse", "tmt_a_raw": "tmt_a_papel_seg", "tmt_b_raw": "tmt_b_papel_seg", "digit_symbol_raw": "digito_simbolo",
    "forward_digit_span_raw": "span_directo", "backward_digit_span_raw": "span_inverso", "clock_drawing_test": "reloj",
    "rt_PART_A": "tiempo_A_seg", "rt_PART_B": "tiempo_B_seg",
    "is_valid_sum_A": "pct_ensayos_validos_A", "is_valid_sum_B": "pct_ensayos_validos_B",
    "distance_difference_from_ideal_PART_A": "desvio_camino_ideal_A", "distance_difference_from_ideal_PART_B": "desvio_camino_ideal_B",
    "hesitation_time_PART_A": "tiempo_duda_A_seg", "hesitation_time_PART_B": "tiempo_duda_B_seg",
    "non_cut_correct_targets_touches_PART_B": "toques_correctos_B",
}
df = datos[list(NOMBRES)].rename(columns=NOMBRES)
df.insert(1, "grupo", datos["group"].map({1: "DCL", 0: "Control"}))
df.insert(3, "sexo", datos["sex"].map({1: "F", 0: "M"}))

print(f"{df.shape[0]} personas y {df.shape[1]} columnas")
df.head()

**Una fila por persona.** Los 20 ensayos de cada participante ya están promediados, así que nunca vamos a tener datos de la misma persona en entrenamiento y en evaluación.

| Tipo | Columnas |
|---|---|
| **Etiqueta** (lo que queremos predecir) | `grupo`: DCL o Control |
| **Demográficas** | `edad`, `sexo`, `anios_educacion` |
| **Neuropsicológicas** (las que se usaron para el diagnóstico) | `mmse`, `tmt_a_papel_seg`, `tmt_b_papel_seg`, `digito_simbolo`, `span_directo`, `span_inverso`, `reloj` |
| **Digitales** (del TMT en la computadora) | `tiempo_A_seg`, `tiempo_B_seg`: tiempo por ensayo · `pct_ensayos_validos_A/B`: porcentaje de ensayos donde llegó al menos a 8 ítems · `desvio_camino_ideal_A/B`: cuánto se aleja el mouse del camino recto · `tiempo_duda_A/B_seg`: tiempo en estado de duda · `toques_correctos_B`: ítems correctos en la parte B |

En el paper usamos unas 100 variables digitales. Acá elegimos 9 fáciles de interpretar.

In [ ]:
DEMOGRAFICAS = ["edad", "sexo_F", "anios_educacion"]
NEUROPSICOLOGICAS = ["mmse", "tmt_a_papel_seg", "tmt_b_papel_seg", "digito_simbolo", "span_directo", "span_inverso", "reloj"]
DIGITALES = ["tiempo_A_seg", "tiempo_B_seg", "pct_ensayos_validos_A", "pct_ensayos_validos_B",
             "desvio_camino_ideal_A", "desvio_camino_ideal_B", "tiempo_duda_A_seg", "tiempo_duda_B_seg", "toques_correctos_B"]

df["sexo_F"] = (df["sexo"] == "F").astype(int)   # los modelos necesitan números
y = (df["grupo"] == "DCL").astype(int)           # 1 = DCL (positivo), 0 = Control
print(df["grupo"].value_counts())

## 2. Mirar los datos antes de modelar
Antes de entrenar cualquier modelo, conviene mirar. ¿Qué variables parecen distintas entre grupos?

In [ ]:
VARIABLES_A_MIRAR = ["tmt_b_papel_seg", "tiempo_B_seg", "pct_ensayos_validos_B", "tiempo_duda_A_seg"]  # ✏️ probá otras columnas

fig, axes = plt.subplots(1, len(VARIABLES_A_MIRAR), figsize=(4 * len(VARIABLES_A_MIRAR), 4))
for ax, var in zip(axes, VARIABLES_A_MIRAR):
    datos = [df.loc[df["grupo"] == g, var] for g in ["Control", "DCL"]]
    ax.boxplot(datos, tick_labels=["Control", "DCL"])
    for i, g in enumerate(["Control", "DCL"]):
        ax.scatter(np.random.normal(i + 1, 0.06, len(datos[i])), datos[i], color=COLORES[g], alpha=0.6, s=15)
    ax.set_title(var)
plt.tight_layout()
plt.show()

Los grupos se superponen mucho: **ninguna variable sola separa perfecto**. Por eso tiene sentido combinar varias en un modelo.

Ahora miremos la relación entre la versión en papel y la digital del TMT-B.

In [ ]:
r = df["tmt_b_papel_seg"].corr(df["tiempo_B_seg"])
for g in ["Control", "DCL"]:
    sub = df[df["grupo"] == g]
    plt.scatter(sub["tmt_b_papel_seg"], sub["tiempo_B_seg"], color=COLORES[g], label=g, alpha=0.7)
plt.xlabel("TMT-B en papel (s)")
plt.ylabel("Tiempo por ensayo, parte B digital (s)")
plt.title(f"Correlación de Pearson r = {r:.2f}")
plt.legend()
plt.show()

> **Recordá:** una correlación significativa no garantiza que una variable sirva para predecir casos nuevos. Eso solo se sabe evaluando en datos que el modelo no vio.

## 3. Aprendizaje no supervisado

Le pedimos a **k-means** que arme grupos **sin decirle el diagnóstico**. Después comparamos sus grupos con el diagnóstico real.

In [ ]:
K = 2  # ✏️ cantidad de grupos que buscamos
VARIABLES_CLUSTER = ["tiempo_B_seg", "pct_ensayos_validos_B"]

X_cl = StandardScaler().fit_transform(df[VARIABLES_CLUSTER])
df["cluster"] = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(X_cl)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df[VARIABLES_CLUSTER[0]], df[VARIABLES_CLUSTER[1]], c=df["cluster"], cmap="viridis")
axes[0].set_title("Grupos que encontró k-means")
for g in ["Control", "DCL"]:
    sub = df[df["grupo"] == g]
    axes[1].scatter(sub[VARIABLES_CLUSTER[0]], sub[VARIABLES_CLUSTER[1]], color=COLORES[g], label=g)
axes[1].set_title("Diagnóstico real")
axes[1].legend()
for ax in axes:
    ax.set_xlabel(VARIABLES_CLUSTER[0]); ax.set_ylabel(VARIABLES_CLUSTER[1])
plt.tight_layout(); plt.show()

pd.crosstab(df["cluster"], df["grupo"])

La tabla cuenta cuántas personas de cada diagnóstico cayeron en cada grupo. **k-means siempre encuentra grupos**, pero no tienen por qué coincidir con lo que nos interesa.

Ahora resumimos las 9 variables digitales en solo 2 con **PCA** (reducción de dimensionalidad) para poder dibujarlas.

In [ ]:
X_pca = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(df[DIGITALES]))
for g in ["Control", "DCL"]:
    m = df["grupo"] == g
    plt.scatter(X_pca[m, 0], X_pca[m, 1], color=COLORES[g], label=g, alpha=0.7)
plt.xlabel("Componente 1"); plt.ylabel("Componente 2"); plt.legend()
plt.title("Las 9 variables digitales resumidas en 2")
plt.show()

## 4. Un primer clasificador y el sobreajuste

Separamos los datos: **75 % para entrenar** y **25 % para evaluar**. La separación es estratificada: mantiene la proporción de DCL y controles en las dos partes.

Usamos una **regresión logística**, un modelo simple que devuelve la probabilidad de DCL.

In [ ]:
SEMILLA = 2  # ✏️ cambia qué personas caen en cada parte

X = df[DIGITALES]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMILLA)

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_train, y_train)

print(f"Entrenamiento: {len(y_train)} personas · Evaluación: {len(y_test)} personas")
print(f"Accuracy en entrenamiento: {modelo.score(X_train, y_train):.2f}")
print(f"Accuracy en evaluación:    {modelo.score(X_test, y_test):.2f}")

`make_pipeline` junta la estandarización y el modelo. Así, la estandarización se calcula **solo con los datos de entrenamiento** y no se filtra información de las personas de evaluación.

**Sobreajuste.** Un árbol de decisión sin límite de profundidad puede memorizar a cada persona de entrenamiento.

In [ ]:
PROFUNDIDAD_MAXIMA = None  # ✏️ probá 1, 2 o 3. None significa sin límite

arbol = DecisionTreeClassifier(max_depth=PROFUNDIDAD_MAXIMA, random_state=0).fit(X_train, y_train)
print(f"Accuracy en entrenamiento: {arbol.score(X_train, y_train):.2f}")
print(f"Accuracy en evaluación:    {arbol.score(X_test, y_test):.2f}")

> **Para pensar:** si evaluáramos con los mismos datos de entrenamiento, ¿qué modelo parecería mejor?
>
> **✏️ Ejercicio:** volvé a la celda de la regresión logística, cambiá `SEMILLA` a 1, 3, 4 y 5, y anotá la accuracy en evaluación. ¿Cuánto cambia? Con solo 19 personas para evaluar, el resultado depende mucho de quién cayó en cada parte. Esa es la motivación de la validación cruzada (sección 8).

## 5. Matriz de confusión y métricas

Miramos qué tipo de errores comete la regresión logística en las personas de evaluación.

In [ ]:
pred = modelo.predict(X_test)
metrics.ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[1, 0], display_labels=["DCL", "Control"], cmap="Greens")
plt.xlabel("Lo que dice el modelo"); plt.ylabel("Real")
plt.show()

VP = int(((y_test == 1) & (pred == 1)).sum()); FN = int(((y_test == 1) & (pred == 0)).sum())
FP = int(((y_test == 0) & (pred == 1)).sum()); VN = int(((y_test == 0) & (pred == 0)).sum())
print(f"VP = {VP}   FN = {FN}   FP = {FP}   VN = {VN}")

Calculamos las métricas **a mano**, con las mismas fórmulas del curso:

In [ ]:
accuracy      = (VP + VN) / (VP + VN + FP + FN)
precision     = VP / (VP + FP) if (VP + FP) > 0 else float("nan")
sensibilidad  = VP / (VP + FN)
especificidad = VN / (VN + FP)
f1            = 2 * precision * sensibilidad / (precision + sensibilidad)
balanced      = (sensibilidad + especificidad) / 2

print(f"Accuracy          = ({VP} + {VN}) / {VP + VN + FP + FN} = {accuracy:.2f}")
print(f"Precisión         = {VP} / ({VP} + {FP}) = {precision:.2f}")
print(f"Sensibilidad      = {VP} / ({VP} + {FN}) = {sensibilidad:.2f}")
print(f"Especificidad     = {VN} / ({VN} + {FP}) = {especificidad:.2f}")
print(f"F1                = {f1:.2f}")
print(f"Balanced accuracy = {balanced:.2f}")

Las mismas métricas con las funciones de scikit-learn, para comprobar:

In [ ]:
print(f"Accuracy          {metrics.accuracy_score(y_test, pred):.2f}")
print(f"Precisión         {metrics.precision_score(y_test, pred):.2f}")
print(f"Sensibilidad      {metrics.recall_score(y_test, pred):.2f}")
print(f"F1                {metrics.f1_score(y_test, pred):.2f}")
print(f"Balanced accuracy {metrics.balanced_accuracy_score(y_test, pred):.2f}")

**La trampa de la accuracy.** En estos datos la clase más frecuente es DCL (41 de 74). Un modelo que dice «DCL» a todo el mundo, sin mirar ninguna variable:

In [ ]:
tonto = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_tonto = tonto.predict(X_test)
print(f"Accuracy          {metrics.accuracy_score(y_test, pred_tonto):.2f}")
print(f"Sensibilidad      {metrics.recall_score(y_test, pred_tonto):.2f}")
print(f"Especificidad     {metrics.recall_score(y_test, pred_tonto, pos_label=0):.2f}")
print(f"Balanced accuracy {metrics.balanced_accuracy_score(y_test, pred_tonto):.2f}")

> **Para pensar:** ¿qué métrica deja en evidencia que este modelo no sirve?

## 6. El umbral de decisión

En realidad el modelo da una **probabilidad**. Para decidir «DCL» o «Control» usamos un umbral: por defecto, 0.5.

In [ ]:
UMBRAL = 0.5  # ✏️ probá 0.3 y 0.7

proba = modelo.predict_proba(X_test)[:, 1]
pred_u = (proba >= UMBRAL).astype(int)
sens = metrics.recall_score(y_test, pred_u)
esp = metrics.recall_score(y_test, pred_u, pos_label=0)
print(f"Umbral {UMBRAL}: sensibilidad = {sens:.2f} · especificidad = {esp:.2f}")

for g, v in [("Control", 0), ("DCL", 1)]:
    plt.hist(proba[y_test.values == v], bins=np.linspace(0, 1, 11), alpha=0.6, color=COLORES[g], label=g)
plt.axvline(UMBRAL, color="black", ls="--")
plt.xlabel("Probabilidad de DCL según el modelo"); plt.ylabel("Personas"); plt.legend()
plt.show()

> Bajar el umbral detecta más pacientes, pero suma falsas alarmas. Subirlo hace lo contrario. El umbral se elige según el costo de cada tipo de error.

## 7. Curva ROC y AUC

La curva ROC muestra **todos los umbrales a la vez**. El AUC resume la curva en un número: 0.5 es azar y 1 es perfecto.

In [ ]:
metrics.RocCurveDisplay.from_predictions(y_test, proba, name="Regresión logística")
plt.plot([0, 1], [0, 1], "k--", label="Azar")
plt.xlabel("Tasa de falsos positivos (1 − especificidad)"); plt.ylabel("Sensibilidad")
plt.legend(); plt.show()

La curva tiene pocos escalones porque solo hay 19 personas en evaluación. Con validación cruzada vamos a usar a las 74.

## 8. Validación cruzada y baselines

Con **validación cruzada de 5 particiones**, cada persona es predicha una vez por un modelo que no la vio. Juntamos todas las predicciones y calculamos las métricas sobre las 74 personas.

Comparamos cuatro conjuntos de variables, como en el paper:

- **Demográficas:** el piso. ¿Cuánto se predice solo con edad, sexo y educación?
- **Digitales:** lo que queremos probar.
- **Digitales + demográficas.**
- **Neuropsicológicas:** el techo. Son las pruebas que se usaron para hacer el diagnóstico.

In [ ]:
CONJUNTOS = {
    "Demográficas": DEMOGRAFICAS,
    "Digitales": DIGITALES,
    "Digitales + demográficas": DIGITALES + DEMOGRAFICAS,
    "Neuropsicológicas": NEUROPSICOLOGICAS,
}
SEMILLA_CV = 0  # ✏️ cambia cómo se reparten las personas en las 5 particiones
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA_CV)

filas = []
for nombre, cols in CONJUNTOS.items():
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    p = cross_val_predict(m, df[cols], y, cv=cv, method="predict_proba")[:, 1]
    filas.append({"Variables": nombre,
                  "AUC": metrics.roc_auc_score(y, p),
                  "Balanced accuracy": metrics.balanced_accuracy_score(y, p >= 0.5),
                  "Sensibilidad": metrics.recall_score(y, p >= 0.5),
                  "Especificidad": metrics.recall_score(y, p >= 0.5, pos_label=0)})
    fpr, tpr, _ = metrics.roc_curve(y, p)
    plt.plot(fpr, tpr, label=f"{nombre} (AUC = {filas[-1]['AUC']:.2f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("Tasa de falsos positivos"); plt.ylabel("Sensibilidad"); plt.legend(fontsize=9)
plt.show()
pd.DataFrame(filas).round(2)

Los números no van a coincidir exactamente con los del paper. Acá usamos menos variables, un solo modelo y una validación cruzada simple. En el paper probamos varios modelos, ajustamos hiperparámetros y usamos validación cruzada anidada.

> **✏️ Ejercicio 1:** cambiá `SEMILLA_CV` a 1, 2, 3 y 4. ¿Entre qué valores se mueve el AUC de las variables digitales? Con 74 personas, el resultado depende de cómo se reparten. Por eso en el paper reportamos intervalos de confianza.
>
> **✏️ Ejercicio 2:** editá la lista `DIGITALES` en la sección 1 (por ejemplo, dejá solo `"pct_ensayos_validos_B"`), ejecutá de nuevo esa celda y esta. ¿Cuánto cambia el AUC?

## 9. ¿Es mejor que el azar?

Con 74 personas, un AUC moderado podría salir por casualidad. El **test de permutaciones** mezcla las etiquetas al azar muchas veces y repite todo el análisis, para ver qué AUC se obtiene sin ninguna relación real.

Tarda unos segundos.

In [ ]:
N_PERMUTACIONES = 200  # ✏️ en el paper usamos 1000

for nombre in ["Demográficas", "Digitales"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    auc_real, aucs_azar, p_valor = permutation_test_score(m, df[CONJUNTOS[nombre]], y, cv=cv, scoring="roc_auc",
                                                         n_permutations=N_PERMUTACIONES, random_state=0)
    print(f"{nombre:13s} AUC = {auc_real:.2f} · p = {p_valor:.3f}")

plt.hist(aucs_azar, bins=25, color="#42B9B2")
plt.axvline(auc_real, color="#DE237B", lw=3, label=f"AUC real (digitales) = {auc_real:.2f}")
plt.xlabel("AUC con etiquetas mezcladas al azar"); plt.ylabel("Repeticiones"); plt.legend()
plt.show()

El p-valor es la proporción de repeticiones al azar que igualan o superan al AUC real.

Con estos datos, las variables digitales superan al azar y las demográficas solas no. En el paper, con todas las variables y modelos ajustados, el p-valor de las digitales fue 0.007.

Ojo: `permutation_test_score` promedia el AUC de cada partición, así que el valor puede diferir un poco del de la sección 8.

## 10. Regresión: predecir el MMSE

Ahora la etiqueta es un **número**: el puntaje del MMSE. Usamos las variables digitales y demográficas, y medimos el error con **MAE** y **RMSE**.

El modelo es un **random forest**: muchos árboles de decisión chicos que votan entre sí.

Como baseline usamos un modelo trivial que siempre predice el promedio del MMSE.

In [ ]:
OBJETIVO = "mmse"  # ✏️ probá "digito_simbolo", "tmt_a_papel_seg" o "tmt_b_papel_seg"

y_reg = df[OBJETIVO]
X_reg = df[DIGITALES + DEMOGRAFICAS]
cv_reg = KFold(n_splits=5, shuffle=True, random_state=0)

bosque = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0)
pred_reg = cross_val_predict(bosque, X_reg, y_reg, cv=cv_reg)
pred_prom = cross_val_predict(DummyRegressor(strategy="mean"), X_reg, y_reg, cv=cv_reg)

for nombre, p in [("Modelo", pred_reg), ("Siempre el promedio", pred_prom)]:
    mae = metrics.mean_absolute_error(y_reg, p)
    rmse = np.sqrt(metrics.mean_squared_error(y_reg, p))
    print(f"{nombre:20s} MAE = {mae:.2f} · RMSE = {rmse:.2f}")

plt.scatter(y_reg, pred_reg, color="#9AC529", edgecolor="black")
lim = [y_reg.min(), y_reg.max()]
plt.plot(lim, lim, "k--", label="Predicción perfecta")
plt.xlabel(f"{OBJETIVO} real"); plt.ylabel(f"{OBJETIVO} predicho"); plt.legend()
plt.show()

> **Para pensar:** ¿el modelo mejora al baseline que siempre predice el promedio? ¿Por cuánto? Un MAE solo tiene sentido comparado con algo.
>
> **Ojo: estos no son los datos completos.** El cuaderno usa 9 de las aproximadamente 100 variables digitales del estudio y un modelo sin ajustar. Por eso acá la mejora frente al baseline es chica. En el paper, con todas las variables y modelos optimizados, la predicción del MMSE sí fue significativa (MAE = 1.45, p = 0.004). No saquen conclusiones sobre el TMT digital a partir de esta sección.
>
> Mirá también el gráfico: las predicciones se amontonan cerca del promedio. Con pocos datos, los modelos tienden a ser «prudentes» y les cuesta predecir los valores extremos.

## 11. Para seguir jugando

Algunas ideas, todas cambiando valores marcados con ✏️:

1. **Umbral (sección 6):** ¿qué umbral elegirías para un screening, donde no queremos que se escapen pacientes?
2. **Clusters (sección 3):** probá `K = 3`. ¿Aparece algún grupo con más DCL?
3. **Variables (sección 1):** ¿cuál es la variable digital que, sola, da el AUC más alto en la sección 8?
4. **Otro puntaje (sección 10):** ¿qué prueba en papel se predice mejor a partir del TMT digital?

**Un recordatorio.** Son 74 personas de un solo centro. Estos modelos sirven para aprender, no para diagnosticar.